# CARO RAG: Rigorous, Fair Evaluation Harness

This notebook provides the interactive execution interface for comparing **CARO RAG** against baseline RAG architectures:
- **B1: Dense Retrieval** (FAISS `bge-m3` top-k + Gemma 3 4B)
- **B2: BM25 Sparse Retrieval** (Inverted index top-k + Gemma 3 4B)
- **B3: Hybrid RRF Retrieval** (Dense + BM25 Reciprocal Rank Fusion, no retrieval LLM calls + Gemma 3 4B)
- **B4: Multi-Query RAG-Fusion** (Query expansion + Hybrid RRF, no router / sufficiency check + Gemma 3 4B)
- **B3-abs: Prompt-Based Abstention** (Hybrid RRF + generation prompt instructed to abstain when evidence is insufficient)
- **CARO: Full Pipeline** (LLM Router, Multi-pass exhaustive candidate search, LLM Evidence Sufficiency Assessor, Grounded Answer Generator)

### Strict Fair-Fight Guarantees
- Same 250k document corpus (`hotpotqa_full_index.faiss` & metadata)
- Same Gemma 3 4B generation model (temperature 0.0)
- Same shared base generation prompt format across all systems
- Same final context window (`MAX_CONTEXT_DOCS = 8`)
- Exact exclusion filter applied identically after over-fetching
- Unbiased external LLM judge (`llama3.1:8b`) for semantic correctness
- NLI cross-encoder (`cross-encoder/nli-deberta-v3-small`) for claim-level faithfulness

In [ ]:
import os
import sys
from pathlib import Path

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from eval.caro_core import CorpusAssets, FAISS_INDEX_PATH, METADATA_PATH, LLM_MODEL_NAME
from eval.runner import run_evaluation
from eval.report import generate_evaluation_report, REPORT_FILE

print("Project Root:", PROJECT_ROOT)
print("Corpus Index:", FAISS_INDEX_PATH)
print("Corpus Meta :", METADATA_PATH)
print("Generator LLM:", LLM_MODEL_NAME)

## 1. Load Shared Assets
Loads the 250,000 FAISS vectors, metadata, embedding model, and inverted index into memory once.

In [ ]:
CorpusAssets.load()
print(f"Loaded {CorpusAssets.total_docs} corpus documents.")

## 2. Execute Smoke Test (N=10 Questions)
Validates all 6 systems across ANSWERABLE, PARTIAL, and UNANSWERABLE sets in ~15-20 minutes.

In [ ]:
smoke_stats = run_evaluation(smoke=True, seed=42)
print("Smoke evaluation finished.")

## 3. Generate and View Evaluation Report
Computes bootstrap 95% confidence intervals, paired hypothesis tests, error analysis, and writes `eval/RESULTS.md`.

In [ ]:
report_md = generate_evaluation_report(seed=42, n_bootstrap=1000)
from IPython.display import display, Markdown
display(Markdown(report_md))

## 4. Full Evaluation Run (Configurable N=500)
The runner is fully resumable; intermediate results are appended to JSONL and skipped if interrupted.

In [ ]:
# To run the full benchmark:
# full_stats = run_evaluation(smoke=False, sample_size=500, seed=42)
# generate_evaluation_report(seed=42, n_bootstrap=1000)
print("Full run cell ready. Run when prepared.")